# 03 · Silver → Gold

Conceito praticado: **Star Schema** com tabelas-ponte (bridge tables)
- `fact_movies_performance` (fato, grão de **um registro por filme lançado**)
- `dim_movies`, `dim_genres`, `dim_people`, `dim_companies`, `dim_reviews` (dimensões com chave substituta)
- `bridge_movie_genre`, `bridge_movie_person`, `bridge_movie_company` (relacionamentos N:N)

Ao final, o Desafio de Analytics responde às perguntas de negócio a partir da Gold.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
catalog = "cinedata_analytics"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

DataFrame[]

## 1. dim_movies
Construída primeiro porque a fato e as bridges usam a sua chave `sk_movie_id`, gerada com `row_number()`.

In [0]:
df_info_silver = spark.table(f"{silver_schema}.tb_info_filmes")

df_dim_movies = (
    df_info_silver
    .withColumn("sk_movie_id", F.row_number().over(Window.orderBy(F.col("id_filme").cast("bigint"))).cast("bigint"))
    .select(
        "sk_movie_id",
        F.col("id_filme").cast("string").alias("id_filme"),
        F.col("titulo").cast("string").alias("titulo"),
        F.col("data_lancamento").cast("date").alias("data_lancamento"),
        F.col("ano_lancamento").cast("int").alias("ano_lancamento"),
        F.col("duracao_minutos").cast("int").alias("duracao_minutos"),
        F.col("idioma_original").cast("string").alias("idioma_original"),
        F.col("status_filme").cast("string").alias("status_filme"),
        F.col("sinopse").cast("string").alias("sinopse"),
    )
)

df_dim_movies.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_movies")

print(f"dim_movies: {df_dim_movies.count()} filmes")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_movies: 97879 filmes


## 2. fact_movies_performance
Grão: **um registro por filme lançado** (`status_filme = 'Lançado'`).
As tabelas financeira e de métricas da Silver podem ter mais de uma linha para o mesmo filme. Para não duplicar o grão nos joins, elas são consolidadas em uma linha por filme antes de entrar na fato:
- **Financeiro**: mantém a linha mais completa do filme (com receita e orçamento preenchidos), para que lucro e valores em BRL continuem coerentes entre si
- **Métricas**: mantém o maior valor válido de cada métrica

In [0]:
df_filmes_lancados = (
    spark.table(f"{gold_schema}.dim_movies")
    .filter(F.col("status_filme") == "Lançado")
    .select("sk_movie_id", "id_filme")
)

janela_financeiro = (
    Window.partitionBy("id_filme")
    .orderBy(F.col("receita_usd").desc_nulls_last(), F.col("orcamento_usd").desc_nulls_last())
)

df_financeiro = (
    spark.table(f"{silver_schema}.tb_financeiro_filmes")
    .withColumn("rn", F.row_number().over(janela_financeiro))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

df_metricas = (
    spark.table(f"{silver_schema}.tb_metricas_engajamento")
    .groupBy("id_filme")
    .agg(
        F.max("popularidade").alias("popularidade"),
        F.max("nota_media_tmdb").alias("nota_media_tmdb"),
        F.max("qtd_votos_tmdb").alias("qtd_votos_tmdb"),
        F.max("nota_media_imdb").alias("nota_media_imdb"),
        F.max("qtd_votos_imdb").alias("qtd_votos_imdb"),
    )
)

df_fact_movies_performance = (
    df_filmes_lancados
    .join(df_financeiro, on="id_filme", how="left")
    .join(df_metricas, on="id_filme", how="left")
    .select(
        F.col("sk_movie_id").cast("bigint").alias("sk_movie_id"),
        F.col("orcamento_usd").cast("decimal(18,2)").alias("orcamento_usd"),
        F.col("receita_usd").cast("decimal(18,2)").alias("receita_usd"),
        F.col("lucro_usd").cast("decimal(18,2)").alias("lucro_usd"),
        F.col("orcamento_brl").cast("decimal(18,2)").alias("orcamento_brl"),
        F.col("receita_brl").cast("decimal(18,2)").alias("receita_brl"),
        F.col("lucro_brl").cast("decimal(18,2)").alias("lucro_brl"),
        F.col("popularidade").cast("double").alias("popularidade"),
        F.col("nota_media_tmdb").cast("double").alias("nota_media_tmdb"),
        F.col("qtd_votos_tmdb").cast("int").alias("qtd_votos_tmdb"),
        F.col("nota_media_imdb").cast("double").alias("nota_media_imdb"),
        F.col("qtd_votos_imdb").cast("int").alias("qtd_votos_imdb"),
    )
)

df_fact_movies_performance.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.fact_movies_performance")

print(f"fact_movies_performance: {df_fact_movies_performance.count()} linhas | "
      f"{df_fact_movies_performance.select('sk_movie_id').distinct().count()} filmes distintos")

fact_movies_performance: 96463 linhas | 96463 filmes distintos


## 3. dim_genres

In [0]:
df_generos_silver = spark.table(f"{silver_schema}.tb_generos")

df_dim_genres = (
    df_generos_silver
    .select("nome_genero").distinct()
    .withColumn("sk_genre_id", F.row_number().over(Window.orderBy("nome_genero")).cast("bigint"))
    .select("sk_genre_id", "nome_genero")
)

df_dim_genres.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_genres")

print(f"dim_genres: {df_dim_genres.count()} gêneros")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_genres: 19 gêneros


## 4. dim_people
Apenas pessoas físicas (Ator, Diretor, Roteirista). Uma mesma pessoa pode aparecer com mais de um `tipo_pessoa`, pois a chave da dimensão é nome + tipo.

In [0]:
df_pessoas_empresas_silver = spark.table(f"{silver_schema}.tb_pessoas_empresas")

df_dim_people = (
    df_pessoas_empresas_silver
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select(F.col("nome_entidade").alias("nome_pessoa"), F.col("tipo_entidade").alias("tipo_pessoa"))
    .distinct()
    .withColumn("sk_person_id", F.row_number().over(Window.orderBy("nome_pessoa", "tipo_pessoa")).cast("bigint"))
    .select("sk_person_id", "nome_pessoa", "tipo_pessoa")
)

df_dim_people.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_people")

print(f"dim_people: {df_dim_people.count()} pessoas")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_people: 417941 pessoas


## 5. dim_companies

In [0]:
df_dim_companies = (
    df_pessoas_empresas_silver
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(F.col("nome_entidade").alias("nome_produtora"))
    .distinct()
    .withColumn("sk_company_id", F.row_number().over(Window.orderBy("nome_produtora")).cast("bigint"))
    .select("sk_company_id", "nome_produtora")
)

df_dim_companies.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_companies")

print(f"dim_companies: {df_dim_companies.count()} produtoras")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_companies: 44464 produtoras


## 6. dim_reviews
Resumo das avaliações de usuários por filme: quantidade de avaliações e nota média arredondada em 2 casas.

In [0]:
df_avaliacoes_silver = spark.table(f"{silver_schema}.tb_avaliacoes_usuarios")
df_movies_lk = spark.table(f"{gold_schema}.dim_movies").select("sk_movie_id", "id_filme")

df_dim_reviews = (
    df_avaliacoes_silver
    .groupBy("id_filme")
    .agg(
        F.count("*").cast("int").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios"),
    )
    .join(df_movies_lk, on="id_filme", how="inner")
    .withColumn("sk_review_id", F.row_number().over(Window.orderBy("sk_movie_id")).cast("bigint"))
    .select("sk_review_id", "sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios")
)

df_dim_reviews.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_reviews")

print(f"dim_reviews: {df_dim_reviews.count()} filmes com avaliações")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_reviews: 27303 filmes com avaliações


## 7. Bridge tables
Ligam a `dim_movies` às dimensões periféricas (gêneros, pessoas e produtoras) sem duplicar o registro da tabela fato.

In [0]:
df_genres_lk = spark.table(f"{gold_schema}.dim_genres")
df_people_lk = spark.table(f"{gold_schema}.dim_people")
df_companies_lk = spark.table(f"{gold_schema}.dim_companies")

df_bridge_movie_genre = (
    df_generos_silver
    .join(df_movies_lk, on="id_filme", how="inner")
    .join(df_genres_lk, on="nome_genero", how="inner")
    .select("sk_movie_id", "sk_genre_id")
    .distinct()
)

df_bridge_movie_person = (
    df_pessoas_empresas_silver
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .withColumnRenamed("nome_entidade", "nome_pessoa")
    .withColumnRenamed("tipo_entidade", "tipo_pessoa")
    .join(df_movies_lk, on="id_filme", how="inner")
    .join(df_people_lk, on=["nome_pessoa", "tipo_pessoa"], how="inner")
    .select("sk_movie_id", "sk_person_id")
    .distinct()
)

df_bridge_movie_company = (
    df_pessoas_empresas_silver
    .filter(F.col("tipo_entidade") == "Produtora")
    .withColumnRenamed("nome_entidade", "nome_produtora")
    .join(df_movies_lk, on="id_filme", how="inner")
    .join(df_companies_lk, on="nome_produtora", how="inner")
    .select("sk_movie_id", "sk_company_id")
    .distinct()
)

df_bridge_movie_genre.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.bridge_movie_genre")
df_bridge_movie_person.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.bridge_movie_person")
df_bridge_movie_company.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.bridge_movie_company")

print(f"bridge_movie_genre: {df_bridge_movie_genre.count()} relações")
print(f"bridge_movie_person: {df_bridge_movie_person.count()} relações")
print(f"bridge_movie_company: {df_bridge_movie_company.count()} relações")

bridge_movie_genre: 131161 relações
bridge_movie_person: 758910 relações
bridge_movie_company: 116607 relações


## 8. Tabela de contexto para o time de IA
Uma linha por filme lançado, com os metadados do filme e de suas dimensões (gêneros, pessoas e produtoras) consolidados em uma coluna de texto `contexto`, pronta para ser usada como entrada de modelos de linguagem.

In [0]:
def campo(rotulo: str, coluna: str):
    """Monta 'Rótulo: valor' apenas quando o valor existe, para não gerar trechos vazios no texto."""
    valor = F.col(coluna).cast("string")
    return F.when(valor.isNotNull() & (valor != ""), F.concat(F.lit(f"{rotulo}: "), valor))

df_generos_filme = (
    spark.table(f"{gold_schema}.bridge_movie_genre")
    .join(spark.table(f"{gold_schema}.dim_genres"), on="sk_genre_id")
    .groupBy("sk_movie_id")
    .agg(F.concat_ws(", ", F.array_sort(F.collect_list("nome_genero"))).alias("generos"))
)

df_pessoas_filme = (
    spark.table(f"{gold_schema}.bridge_movie_person")
    .join(spark.table(f"{gold_schema}.dim_people"), on="sk_person_id")
    .groupBy("sk_movie_id")
    .agg(
        F.concat_ws(", ", F.collect_list(F.when(F.col("tipo_pessoa") == "Diretor", F.col("nome_pessoa")))).alias("diretores"),
        F.concat_ws(", ", F.collect_list(F.when(F.col("tipo_pessoa") == "Roteirista", F.col("nome_pessoa")))).alias("roteiristas"),
        F.concat_ws(", ", F.collect_list(F.when(F.col("tipo_pessoa") == "Ator", F.col("nome_pessoa")))).alias("elenco"),
    )
)

df_produtoras_filme = (
    spark.table(f"{gold_schema}.bridge_movie_company")
    .join(spark.table(f"{gold_schema}.dim_companies"), on="sk_company_id")
    .groupBy("sk_movie_id")
    .agg(F.concat_ws(", ", F.collect_list("nome_produtora")).alias("produtoras"))
)

df_contexto_ia = (
    spark.table(f"{gold_schema}.dim_movies")
    .filter(F.col("status_filme") == "Lançado")
    .select("sk_movie_id", "titulo", "ano_lancamento", "duracao_minutos", "sinopse")
    .join(spark.table(f"{gold_schema}.fact_movies_performance")
          .select("sk_movie_id", "nota_media_imdb", "popularidade"), on="sk_movie_id", how="left")
    .join(df_generos_filme, on="sk_movie_id", how="left")
    .join(df_pessoas_filme, on="sk_movie_id", how="left")
    .join(df_produtoras_filme, on="sk_movie_id", how="left")
    .withColumn("contexto", F.concat_ws(". ",
        campo("Título", "titulo"),
        campo("Ano de lançamento", "ano_lancamento"),
        campo("Duração em minutos", "duracao_minutos"),
        campo("Gêneros", "generos"),
        campo("Direção", "diretores"),
        campo("Roteiro", "roteiristas"),
        campo("Elenco", "elenco"),
        campo("Produtoras", "produtoras"),
        campo("Nota média IMDb", "nota_media_imdb"),
        campo("Sinopse", "sinopse"),
    ))
    .select("sk_movie_id", "titulo", "generos", "diretores", "roteiristas", "elenco",
            "produtoras", "nota_media_imdb", "popularidade", "sinopse", "contexto")
)

df_contexto_ia.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.tb_contexto_ia")

display(df_contexto_ia.select("titulo", "contexto").limit(5))

titulo,contexto
Rings,"Título: Rings. Ano de lançamento: 2017. Duração em minutos: 102. Gêneros: Horror. Direção: F. Javier Gutiérrez. Roteiro: Akiva Goldsman, Jacob Estes, David Loucka, Kôji Suzuki. Elenco: Laura Slade Wiggins, Aimee Teegarden, Patrick Walker, Vincent D'onofrio, Bonnie Morgan, Alex Roe, Zach Roerig, Chuck David Willis, Matilda Lutz, Johnny Galecki. Produtoras: Macari/edelstein, Paramount, Waddieish Claretrap, Parkes+macdonald Image Nation, Vertigo Entertainment. Sinopse: \Julia becomes worried about her boyfriend Holt when he explores the dark urban legend of a mysterious videotape said to kill the watcher seven days after viewing. She sacrifices herself to save her boyfriend and in doing so makes a horrifying discovery: there is a """"\""""movie within the movie""""\"""" that no one has ever seen before.""""""""First you watch it. Then you die."
Mixtape,"Título: Mixtape. Ano de lançamento: 2021. Duração em minutos: 94. Gêneros: Comedy, Family. Direção: Valerie Weiss. Roteiro: Stacey Menear. Elenco: Kiefer O'reilly, Olga Petsa, Jackson Rathbone, Gemma Brooke Allen, Julie Bowen, Audrey Hsieh, Lucas Yao, Nick Thune, Anthony Timpano. Produtoras: Maven Screen Media, Jim Wedaa Productions. Nota média IMDb: 6.6"
Grizzly II: Revenge,Título: Grizzly II: Revenge. Ano de lançamento: 2020. Duração em minutos: 74. Direção: English. Sinopse: \All hell breaks loose when a giant grizzly
Billy Joel - Live at Yankee Stadium,"Título: Billy Joel - Live at Yankee Stadium. Ano de lançamento: 2022. Duração em minutos: 86. Gêneros: Music. Direção: Jon Small. Roteiro: Martin Torgoff. Elenco: Tommy Byrnes, Billy Joel, David Brown, Jeffrey Jacobs, Schuyler Deale, Mark Rivera, Liberty Devitto. Produtoras: Columbia Music Video. Nota média IMDb: 7.8. Sinopse: Billy Joel plays his greatest hits in the Big Apple."
Bad Boys for Life,"Título: Bad Boys for Life. Ano de lançamento: 2020. Duração em minutos: 124. Gêneros: Action, Crime, Thriller. Direção: Adil El Arbi, Bilall Fallah. Roteiro: Peter Craig, Chris Bremner, Joe Carnahan, George Gallo. Elenco: Alexander Ludwig, Nicky Jam, Kate Del Castillo, Martin Lawrence, Will Smith, Jacob Scipio, Joe Pantoliano, Charles Melton, Vanessa Hudgens, Paola Nuñez. Produtoras: 2.0 Entertainment, Columbia Pictures, Don Simpson/jerry Bruckheimer Films, Overbrook Entertainment. Nota média IMDb: 6.5. Sinopse: Marcus and Mike are forced to confront new threats, career changes, and midlife crises as they join the newly created elite team AMMO of the Miami police department to take down the ruthless Armando Armas, the vicious leader of a Miami drug cartel."


## 9. Desafio de Analytics
Perguntas de negócio respondidas a partir das tabelas Gold.

In [0]:
df_fact = spark.table(f"{gold_schema}.fact_movies_performance")
df_dim_movies = spark.table(f"{gold_schema}.dim_movies")
df_dim_genres = spark.table(f"{gold_schema}.dim_genres")
df_dim_people = spark.table(f"{gold_schema}.dim_people")
df_dim_companies = spark.table(f"{gold_schema}.dim_companies")
df_bridge_genre = spark.table(f"{gold_schema}.bridge_movie_genre")
df_bridge_person = spark.table(f"{gold_schema}.bridge_movie_person")
df_bridge_company = spark.table(f"{gold_schema}.bridge_movie_company")

In [0]:
def formatar_moeda(coluna: str, simbolo: str):
    """Formata o valor no padrão brasileiro para exibição: R$ 1.234.567,89"""
    return F.concat(F.lit(f"{simbolo} "), F.translate(F.format_number(F.col(coluna), 2), ",.", ".,"))

### 1. Qual é a receita total (em R$) somada de todos os filmes da base?

In [0]:
display(
    df_fact
    .agg(F.sum("receita_brl").alias("receita_total_brl"))
    .select(formatar_moeda("receita_total_brl", "R$").alias("receita_total"))
)

receita_total
"R$ 807.278.012.999,64"


### 2. Quais são os 5 filmes com maior popularidade?

In [0]:
df_top_popularidade = (
    df_fact
    .join(df_dim_movies, on="sk_movie_id")
    .filter(F.col("popularidade").isNotNull())
    .orderBy(F.desc("popularidade"))
    .select("titulo", "popularidade")
    .limit(5)
)

display(df_top_popularidade)

titulo,popularidade
blue beetle,2994.357
Gran Turismo,2680.593
The Nun II,1692.778
Meg 2: The Trench,1567.273
retribution,1547.22


### 3. Quantos filmes cada gênero possui?

In [0]:
df_filmes_por_genero = (
    df_bridge_genre
    .join(df_dim_genres, on="sk_genre_id")
    .groupBy("nome_genero")
    .agg(F.countDistinct("sk_movie_id").alias("qtd_filmes"))
    .orderBy(F.desc("qtd_filmes"))
)

display(df_filmes_por_genero)

nome_genero,qtd_filmes
Drama,30457
Documentary,18612
Comedy,17348
Thriller,9424
Horror,9163
Romance,6996
Action,5541
Crime,4320
Animation,4159
TV Movie,3676


Databricks visualization. Run in Databricks to view.

### 4. Para os 10 filmes de maior receita, mostre título, receita (em US$ e R$) e a posição no ranking

In [0]:
janela_receita = Window.orderBy(F.desc("receita_usd"))

df_top_receita = (
    df_fact
    .filter(F.col("receita_usd").isNotNull())
    .join(df_dim_movies, on="sk_movie_id")
    .withColumn("posicao", F.rank().over(janela_receita))
    .filter(F.col("posicao") <= 10)
    .select(
        "posicao",
        "titulo",
        formatar_moeda("receita_usd", "US$").alias("receita_usd"),
        formatar_moeda("receita_brl", "R$").alias("receita_brl"),
    )
    .orderBy("posicao")
)

display(df_top_receita)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


posicao,titulo,receita_usd,receita_brl
1,Avengers: Endgame,"US$ 2.800.000.000,00","R$ 13.913.760.000,00"
2,Avatar: The Way of Water,"US$ 2.320.250.281,00","R$ 11.529.787.696,35"
3,AVENGERS: INFINITY WAR,"US$ 2.052.415.039,00","R$ 10.198.860.811,80"
4,spider-man: no way home,"US$ 1.921.847.111,00","R$ 9.550.042.663,98"
5,The Lion King,"US$ 1.663.075.401,00","R$ 8.264.154.282,65"
6,Top Gun: Maverick,"US$ 1.488.732.821,00","R$ 7.397.811.134,11"
7,Barbie,"US$ 1.428.545.028,00","R$ 7.098.725.953,14"
8,The Super Mario Bros. Movie,"US$ 1.355.725.263,00","R$ 6.736.869.976,90"
9,Black Panther,"US$ 1.349.926.083,00","R$ 6.708.052.691,64"
10,Star Wars: The Last Jedi,"US$ 1.332.698.830,00","R$ 6.622.447.026,04"


### Data limite para as perguntas 5 e 6
Data de lançamento mais recente entre os filmes lançados, ignorando datas futuras.

In [0]:
data_limite = (
    df_dim_movies
    .filter((F.col("status_filme") == "Lançado") & (F.col("data_lancamento") <= F.current_date()))
    .agg(F.max("data_lancamento"))
    .first()[0]
)

print(f"Data limite: {data_limite}")

Data limite: 2026-02-19


### 5. Qual ator teve a maior quantidade de participações nos filmes lançados nos últimos 2 anos?

In [0]:
df_filmes_2_anos = df_dim_movies.filter(
    (F.col("status_filme") == "Lançado") &
    F.col("data_lancamento").between(F.add_months(F.lit(data_limite), -24), F.lit(data_limite))
)

df_ator_top = (
    df_filmes_2_anos.select("sk_movie_id")
    .join(df_bridge_person, on="sk_movie_id")
    .join(df_dim_people.filter(F.col("tipo_pessoa") == "Ator"), on="sk_person_id")
    .groupBy("nome_pessoa")
    .agg(F.countDistinct("sk_movie_id").alias("qtd_participacoes"))
    .orderBy(F.desc("qtd_participacoes"))
    .limit(5)
)

display(df_ator_top)

nome_pessoa,qtd_participacoes
Kevin Hart,64
John Travolta,59
Alex Dauphin,59
Stacy Hall,59
Olivia Park,59


### 6. Qual produtora teve o maior lucro nos últimos 5 anos?
O lucro de um filme é atribuído a cada uma das suas produtoras.

In [0]:
df_filmes_5_anos = df_dim_movies.filter(
    (F.col("status_filme") == "Lançado") &
    F.col("data_lancamento").between(F.add_months(F.lit(data_limite), -60), F.lit(data_limite))
)

df_produtora_top = (
    df_filmes_5_anos.select("sk_movie_id")
    .join(df_fact, on="sk_movie_id")
    .join(df_bridge_company, on="sk_movie_id")
    .join(df_dim_companies, on="sk_company_id")
    .groupBy("nome_produtora")
    .agg(
        F.sum("lucro_usd").alias("lucro_total_usd"),
        F.sum("lucro_brl").alias("lucro_total_brl"),
    )
    .filter(F.col("lucro_total_usd").isNotNull())
    .orderBy(F.desc("lucro_total_usd"))
    .limit(5)
    .select(
        "nome_produtora",
        formatar_moeda("lucro_total_usd", "US$").alias("lucro_total_usd"),
        formatar_moeda("lucro_total_brl", "R$").alias("lucro_total_brl"),
    )
)

display(df_produtora_top)

nome_produtora,lucro_total_usd,lucro_total_brl
Universal Pictures,"US$ 5.772.329.679,00","R$ 28.683.860.640,89"
Marvel Studios,"US$ 4.953.462.823,00","R$ 24.614.747.460,07"
Columbia Pictures,"US$ 3.662.050.755,00","R$ 18.197.462.611,75"
Pascal Pictures,"US$ 2.701.952.454,00","R$ 13.426.542.134,42"
Illumination,"US$ 2.431.353.473,00","R$ 12.081.881.678,03"
